In [0]:
# 410-*-.json finden und das erste Paar drucken.

import json
from pathlib import Path
name = "data/sample/410/410_DE_hour_1787522400000.json"
candidates = [Path.cwd() / name, Path.cwd().parent / name]
path = next(p for p in candidates if p.exists())
print(path)
with open(path) as f:
    data = json.load(f)
print(list(data.keys()))
print("hours", len(data["series"]))
print("first row", data["series"][0])

In [0]:
# jedes Paar wird eine Zeile und landet in bronze.smard_raw. Das ist Ingest.
# Zeile --> (filter_id, metric=load, event_time_ms, value_mwh) 
from pyspark.sql import functions as F

rows = [(410, "load", int(ts), val) for ts, val in data["series"]]
df = spark.createDataFrame(rows, ["filter_id", "metric", "event_time_ms", "value_mwh"])
df = df.withColumn("source_file", F.lit(str(path)))
df.show(5)

df.write.mode("overwrite").saveAsTable("bronze.smard_raw")
print("rows", df.count())

In [0]:
# Berliner Uhrzeit, Lücken markieren, Duplikate weg, silver.electricity_hourly. Das ist Putzen.

silver = (
    df.withColumn(
        "event_time",
        F.from_utc_timestamp(F.to_timestamp(F.col("event_time_ms") / 1000), "Europe/Berlin"),
    )
    .withColumn("is_missing_value", F.col("value_mwh").isNull())
    .dropDuplicates(["metric", "event_time"])
)

silver.show(5)
silver.write.mode("overwrite").saveAsTable("silver.electricity_hourly")
print("rows", silver.count())

In [0]:
# Stunden zu Tagen, min/schnitt/max, gold.daily_load. Das ist die Report-Tabelle.

gold = (
    silver.groupBy(F.to_date("event_time").alias("day"))
    .agg(
        F.count("*").alias("hours"),
        F.sum(F.when(F.col("is_missing_value"), 1).otherwise(0)).alias("missing_hours"),
        F.avg("value_mwh").alias("avg_load_mwh"),
        F.min("value_mwh").alias("min_load_mwh"),
        F.max("value_mwh").alias("max_load_mwh"),
    )
)

gold.show()
gold.write.mode("overwrite").saveAsTable("gold.daily_load")
print("days", gold.count())

In [0]:


name = "data/sample/4067/4067_DE_hour_1787522400000.json"
candidates = [Path.cwd() / name, Path.cwd().parent / name]
wind_path = next(p for p in candidates if p.exists())

with open(wind_path) as f:
    wind = json.load(f)

rows = [(4067, "wind_onshore", int(ts), val) for ts, val in wind["series"]]
wind_df = spark.createDataFrame(rows, ["filter_id", "metric", "event_time_ms", "value_mwh"])
wind_df = wind_df.withColumn("source_file", F.lit(str(wind_path)))
wind_df.show(5)

wind_df.write.mode("append").saveAsTable("bronze.smard_raw")
print("bronze rows now", spark.table("bronze.smard_raw").count())

In [0]:

bronze = spark.table("bronze.smard_raw")

silver = (
    bronze.withColumn(
        "event_time",
        F.from_utc_timestamp(F.to_timestamp(F.col("event_time_ms") / 1000), "Europe/Berlin"),
    )
    .withColumn("is_missing_value", F.col("value_mwh").isNull())
    .dropDuplicates(["metric", "event_time"])
)

silver.groupBy("metric").count().show()
silver.write.mode("overwrite").saveAsTable("silver.electricity_hourly")
print("silver rows", silver.count())

In [0]:

hourly = (
    spark.table("silver.electricity_hourly")
    .groupBy("event_time")
    .pivot("metric", ["load", "wind_onshore"])
    .agg(F.first("value_mwh"))
)

gold = (
    hourly.withColumn("wind_share", F.col("wind_onshore") / F.col("load"))
    .groupBy(F.to_date("event_time").alias("day"))
    .agg(
        F.avg("load").alias("avg_load_mwh"),
        F.avg("wind_onshore").alias("avg_wind_mwh"),
        F.avg("wind_share").alias("avg_wind_share"),
        F.max("wind_share").alias("max_wind_share"),
    )
)

gold.show()
gold.write.mode("overwrite").saveAsTable("gold.daily_energy_mix")

In [0]:
hourly = (
    spark.table("silver.electricity_hourly")
    .groupBy("event_time")
    .pivot("metric", ["load", "wind_onshore"])
    .agg(F.first("value_mwh"))
    .filter(F.col("load").isNotNull())
)

gold = (
    hourly.withColumn("wind_share", F.col("wind_onshore") / F.col("load"))
    .groupBy(F.to_date("event_time").alias("day"))
    .agg(
        F.count("*").alias("hours"),
        F.avg("load").alias("avg_load_mwh"),
        F.avg("wind_onshore").alias("avg_wind_mwh"),
        F.avg("wind_share").alias("avg_wind_share"),
        F.max("wind_share").alias("max_wind_share"),
    )
)

gold.show()
gold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.daily_energy_mix")
